In [1]:
import scoop
from scoop import Person
from scoop.configuration.distribution_base import LabelledDistribution, NumericDistribution
from scoop.utils import concept_mapper, measurement_mapper, observation_mapper
from scoop.utils.config_loader import load_config
from scoop.schema.person import person_from_config
from pathlib import Path

In [2]:
concept_mapper('sex')('male')

8507

In [3]:
smoking_field = observation_mapper("smoking_history")
weight_field = measurement_mapper("weight")

In [4]:
l = LabelledDistribution(probs={'A': 0.2, 'B': 0.5, 'C': 0.3})
[l.sample() for _ in range(15)]

['B', 'B', 'B', 'C', 'B', 'A', 'C', 'B', 'B', 'C', 'B', 'C', 'B', 'A', 'B']

In [5]:
n = NumericDistribution(mean=10, sd=2)
[n.sample() for _ in range(10)]

[10.036458151350296,
 7.157290467951627,
 10.399824822425717,
 11.045938140168515,
 10.835182415725823,
 12.316169696023898,
 7.875927540786926,
 9.364416445315317,
 8.087497291329672,
 7.951238637933345]

In [6]:
n2 = NumericDistribution(mean=10, sd=2, min_value=5, max_value=15, as_int=True)
[n2.sample() for _ in range(10)]

[10, 8, 9, 9, 8, 8, 9, 10, 8, 8]

In [7]:
distributions_path = Path('../scoop/distributions')

c = load_config(distributions_path / 'person_distributions.yaml')

In [8]:
person = person_from_config(c)

print(person)

person_id=None race_concept_id=0 ethnicity_concept_id=0 sex='female' age=63 smoking_history='never' ecog=3 weight=97.82059357316581 variant='none'


In [9]:
measurement_mapper['ecog']

In [10]:
person.model_dump()

{'person_id': 1,
 'race_concept_id': 0,
 'ethnicity_concept_id': 0,
 'year_of_birth': 1962,
 'measurements': [Measurement(person_id=1, measurement_id=None, measurement_concept_id=4099154, measurement_date=datetime.date(2025, 10, 30), value_as_concept_id=None, value_as_number=97.82059357316581, source_value=None),
  Measurement(person_id=1, measurement_id=None, measurement_concept_id=36305384, measurement_date=datetime.date(2025, 10, 30), value_as_concept_id=None, value_as_number=3.0, source_value=None),
  Measurement(person_id=1, measurement_id=None, measurement_concept_id=4180182, measurement_date=datetime.date(2025, 10, 30), value_as_concept_id=0, value_as_number=None, source_value=None)],
 'observations': [Observation(person_id=1, observation_id=None, observation_concept_id=43054909, observation_date=datetime.date(2025, 10, 30), value_as_concept_id=45879404, value_as_number=None, source_value=None)],
 'gender_concept_id': 8532}

In [11]:
from scoop.configuration.conditional_distribution import ConditionalMatrixDistribution

In [28]:
# we can override the distributions to test/create the case we want as well
p = Person(variant="abcd", sex="female", age=70)

In [29]:
dist = ConditionalMatrixDistribution.from_yaml(Path('../scoop/distributions/person_conditional.yaml'), "morphology")

l = []
for _ in range(100):
    l.append(dist.sample({"person": p}))

In [30]:
len(l), len([ll for ll in l if ll == "adenocarcinoma"])

(100, 91)

In [14]:
from scoop.configuration.conditional_distribution import combine_prob_vectors

In [ ]:
[a * b for a, b in zip(combined, vec)]

In [ ]:
mode = "multiply"
if not conditional_vectors:
    raise ValueError("No conditional vectors provided for combination.")
if mode == "multiply":
    combined = [1.0] * len(conditional_vectors[0])
    for vec in conditional_vectors:
        combined = [a * b for a, b in zip(combined, vec)]
elif mode == "average":
    n = len(conditional_vectors)
    combined = [sum(col) / n for col in zip(*conditional_vectors)]
else:
    raise ValueError(f"Unknown combination mode: {mode}")
total = sum(combined)
    #return [v / total for v in combined]


In [ ]:
from typing import Any
from functools import reduce

def get_attr_path(obj: Any, path: str, default=None):
    try:
        return reduce(getattr, path.split("."), obj)
    except AttributeError:
        return default

In [ ]:
conditional_vectors = []
context = {'person': p}
for obj_name, attributes in dist.class_probs.items():
    obj = context.get(obj_name, None)
    if obj is not None:

        for attr_name, table in attributes.items():
            val = getattr(obj, attr_name, None)
            probs = table.get(val, table.get("default"))
            if probs:
                conditional_vectors.append(probs)
# sample
# 
# return random.choices(labels, weights=weights, k=1)[0]


In [ ]:
combined = combine_prob_vectors(conditional_vectors, dist.combination)

In [ ]:
labels, weights = zip(*zip(dist.labels, combined))

In [ ]:
conditional_vectors

In [ ]:
p.variant, p.sex

In [ ]:
combined

In [ ]:
0.9*0.85

In [ ]:
combined

In [ ]:
get_attr_path(p, "person")

In [ ]:
p.__class__